# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Data published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and field `@id`s. This will help in selecting which data to extract in subsequent steps.
We access all record sets in the metadata and display their `@id` and relevant field information.

In [ ]:
# Display all available RecordSets and their Fields by @id
print("Available RecordSets (by @id):\n")
record_set_ids = []
for rset in metadata.record_sets:
    print(f"RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    record_set_ids.append(rset.id)
    print("  Fields:")
    for field in rset.fields:
        print(f"    - name: {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

# Show record example for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Example record for RecordSet '{first_rs}':")
    for i, row in enumerate(dataset.records(record_set=first_rs)):
        pprint(row)
        if i >= 1:
            break

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. All entities must be referenced by their `@id`.

Below, we'll load all record sets discovered above as separate DataFrames for further exploration.

In [ ]:
# Extract data from each record set and load as pandas DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet '@id': {rs_id}")

# Preview the main survey data (use the record set that contains demographic and survey fields)
# This will typically have fields like 'age', 'gad7_score', 'phq9_score', 'gender', etc.
main_rs_id = None
target_fields = {'age', 'gad7_score', 'phq9_score', 'gender'}
for rs_id, df in dataframes.items():
    if target_fields.intersection(set(df.columns)):
        main_rs_id = rs_id
        break

if main_rs_id:
    print(f"\nMain survey record set is: {main_rs_id}")
    print(f"Fields: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("Could not identify a main survey record set. Inspect the DataFrames above.")

## 4. Exploratory Data Analysis (EDA)
Let's process the main survey record set, selecting a numeric mental health score field (such as PHQ-9 or GAD-7 score) for analysis. We'll demonstrate filtering, normalizing, and grouping based on another key attribute (e.g., gender).

In [ ]:
# --- Set these IDs based on your record set/field naming --- #
# These must be string names as they appear in the DataFrame columns (from the schema @id)

# For illustration, let's use 'phq9_score' (Patient Health Questionnaire, depression score) and group by 'gender'.
numeric_field_id = 'phq9_score'          # Example field @id
group_field_id = 'gender'                # Example field @id
record_set_id = main_rs_id               # Set from extraction step

if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    # Ensure numeric type for the field (if there are missing values or type issues)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the score
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field, e.g. gender
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)
else:
    print(f"Field '{numeric_field_id}' not found in DataFrame for RecordSet '@id'={record_set_id}")

## 5. Visualization
Let's plot the distribution of the PHQ-9 depression scores, and show mean PHQ-9 score by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (PHQ-9) scores")
    plt.xlabel("PHQ-9 Score")
    plt.ylabel("Count")
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(6, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
We've loaded the FAIR² Mental Health Survey Dataset with `mlcroissant` via Croissant schema, reviewed its structure, and performed basic exploratory analysis of the PHQ-9 depression score. You can extend this workflow for other indicators such as GAD-7 and PSQ, or examine demographic breakdowns and relationships within the data using the record set and field `@id`s throughout.

**Next steps** might include more sophisticated visualizations, statistical hypothesis testing, or integrating other open datasets with Croissant schemas.
